# Lezione 56 — Estrazione strutturata con Gemma + LoRA

> **Modulo:** capstone · **Tempo stimato:** 25 minuti
> **Prerequisiti:** Lezioni 36 (output strutturato), 41 (LoRA su Gemma).
>
> Obiettivo pratico unico: aggiungere al lab l'estrazione delle **relazioni** con
> un modello open adattato via LoRA, con **fallback a regole** che rende il
> componente sempre funzionante.
>
> ⚠️ **Nota di ambiente**: le celle Gemma sono **guardate** e saltate qui (pesi
> gated); il fallback e' pienamente eseguibile. Vedi `course/research_gaps.md`.

## Teoria essenziale

L'ultimo pezzo del record sono le `relations`: triple `(source, type, target)`,
es. `(Marco, visited, Glasgow)`. Un modello Gemma adattato con LoRA (Lezione 41)
e vincolato a output strutturato (Lezione 36) le estrae. Ma il lab deve
funzionare **anche senza modello**: usiamo un estrattore a regole (verbo tra due
entita') come fallback, e il modello come opzione di qualita' superiore.

In [1]:
GEMMA_AVAILABLE = False
_motivo = ""
try:
    import keras  # noqa: F401
    import keras_hub  # noqa: F401
    GEMMA_AVAILABLE = True
except Exception as exc:  # noqa: BLE001
    _motivo = f"{type(exc).__name__}"
print("KerasHub disponibile." if GEMMA_AVAILABLE
      else f"KerasHub/Gemma NON disponibile ({_motivo or 'assente'}); celle modello saltate.")

KerasHub/Gemma NON disponibile (ModuleNotFoundError); celle modello saltate.


In [2]:
SCHEMA = ("Extract relations as JSON list of {source, type, target} from: ")
if GEMMA_AVAILABLE:
    import keras_hub
    gemma = keras_hub.models.GemmaCausalLM.from_preset("gemma_2b_en")
    gemma.backbone.enable_lora(rank=4)     # adattamento efficiente (Lezione 41)
    print(gemma.generate(SCHEMA + "Marco visited Glasgow.", max_length=64))
else:
    print("[saltato] gemma.generate(...) con adapter LoRA")
    print("Con il modello, estrarrebbe le relazioni in JSON.")

[saltato] gemma.generate(...) con adapter LoRA
Con il modello, estrarrebbe le relazioni in JSON.


In [3]:
import re

VERBI = {"visited": "visited", "met": "met", "likes": "likes",
         "prefers": "prefers", "works": "works_at", "lives": "lives_in"}

def estrai_entita(testo):
    return [w for w in re.findall(r"\b[A-Z][a-zA-Z]+\b", str(testo)) if w not in {"The", "A"}]

def estrai_relazioni_regole(testo):
    ent = estrai_entita(testo)
    rels = []
    for v_sup, v_norm in VERBI.items():
        if v_sup in testo.lower() and len(ent) >= 2:
            rels.append({"source": ent[0], "type": v_norm, "target": ent[1]})
            break
    return rels

def estrai_relazioni(testo, usa_modello=GEMMA_AVAILABLE):
    # con il modello si userebbe Gemma+LoRA; qui il fallback a regole
    return estrai_relazioni_regole(testo)

rel = estrai_relazioni("Marco visited Glasgow with his son.")
assert rel and rel[0]["source"] == "Marco" and rel[0]["target"] == "Glasgow"
print("relazioni (fallback a regole):", rel)

relazioni (fallback a regole): [{'source': 'Marco', 'type': 'visited', 'target': 'Glasgow'}]


## Riepilogo (max 8 punti)

1. Ultimo pezzo del record: le `relations` (source, type, target).
2. Gemma + LoRA (Lezione 41) + output strutturato (Lezione 36) le estrae.
3. Il lab deve funzionare **anche senza modello**: fallback a regole.
4. Il fallback cerca un verbo noto tra due entita'.
5. Il modello e' l'opzione di qualita' superiore, non un requisito.
6. Le celle Gemma sono guardate; il fallback e' eseguibile.
7. Cosi' il componente relazioni e' sempre disponibile.
8. Completa i campi del record del progetto.

## Quiz

1. Perche' avere un fallback a regole per le relazioni?
2. Cosa rappresenta una relazione (source, type, target)?
3. Perche' vincolare il modello a output strutturato (Lezione 36)?

*(Risposte: 1. per far funzionare il lab senza GPU/modello e come rete di
sicurezza; 2. un arco del grafo di conoscenza, es. Marco -visited-> Glasgow;
3. per ottenere JSON validabile invece di testo libero.)*